In [ ]:
import os, json, torch, warnings, time, shutil, random, csv
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from pathlib import Path
import torch.nn as nn
warnings.filterwarnings('ignore')

# ── Install Dependencies ──────────────────────────────────────
!pip install -q kagglehub timm
import kagglehub
import timm  # timm has EfficientNet-B0 built-in, no effdet needed

# ── Download Dataset ──────────────────────────────────────────
RAW = Path(kagglehub.dataset_download('imsparsh/deepweeds'))
print(f"Downloaded -> {RAW}")
print(f"Contents   -> {list(RAW.iterdir())[:5]}")

label_files = list(RAW.rglob('labels.csv'))
img_dirs    = [f for f in RAW.rglob('*') if f.is_dir() and f.name.lower() in ['images','imgs']]
LABEL_CSV   = label_files[0]
IMG_DIR     = img_dirs[0] if img_dirs else RAW
print(f"Labels : {LABEL_CSV}")
print(f"Images : {IMG_DIR}")

# ── Config ────────────────────────────────────────────────────
EPOCHS  = 20
BATCH   = 32
LR      = 1e-3
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
NC      = 9
RF      = 'results_efficientdet_d0.json'
CLASSES = ['Chinee_Apple','Lantana','Parkinsonia','Parthenium',
           'Prickly_Acacia','Rubber_Vine','Siam_Weed','Snake_Weed','Negative']
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ── Prepare Dataset ───────────────────────────────────────────
OUT = Path('/kaggle/working/deepweeds')
if not (OUT/'train').exists():
    rows = list(csv.DictReader(open(LABEL_CSV)))
    random.seed(42); random.shuffle(rows)
    sp = int(len(rows)*0.8)
    splits = {'train': rows[:sp], 'val': rows[sp:]}
    for s, data in splits.items():
        for cls in CLASSES:
            (OUT/s/cls).mkdir(parents=True, exist_ok=True)
        for r in data:
            fname = r['Filename'].strip()
            if not fname.endswith('.jpg'): fname += '.jpg'
            cls = CLASSES[int(r['Label'])]
            src = IMG_DIR / fname
            if not src.exists():
                found = list(RAW.rglob(fname))
                src   = found[0] if found else None
            if src and Path(src).exists():
                shutil.copy(src, OUT/s/cls/fname)
    print(f'Split -> train:{sp}  val:{len(rows)-sp}')
else:
    print('Already prepared')

CLS_PATH = str(OUT)

# ── Data Loaders ──────────────────────────────────────────────
t_start = time.time()
print("\n[Data Loading] Building datasets & loaders...")

# EfficientDet-D0 standard input = 512x512
tfm_tr = T.Compose([
    T.Resize((512,512)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(0.3,0.3,0.3),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tfm_vl = T.Compose([
    T.Resize((512,512)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

train_ds = ImageFolder(CLS_PATH+'/train', tfm_tr)
val_ds   = ImageFolder(CLS_PATH+'/val',   tfm_vl)

tr = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
vl = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

t_end = time.time()
print(f"[Data Loading] Train samples : {len(train_ds)}")
print(f"[Data Loading] Val   samples : {len(val_ds)}")
print(f"[Data Loading] Train batches : {len(tr)}")
print(f"[Data Loading] Val   batches : {len(vl)}")
print(f"[Data Loading] Classes       : {train_ds.classes}")
print(f"[Data Loading] Time taken    : {t_end - t_start:.2f} seconds")
print(f"[Data Loading] Done ✓\n")

# ── Model (EfficientDet-D0 via timm — NO effdet needed) ───────
class EfficientDetD0Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # EfficientNet-B0 is the exact backbone used in EfficientDet-D0
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=True,
            num_classes=0,       # strip original classifier
            global_pool='avg'
        )
        in_features = self.backbone.num_features  # 1280

        # BiFPN-inspired classification head
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        # Kaiming init for all linear layers
        for layer in self.head.modules():
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, mode='fan_out')
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

    def forward(self, x):
        x = self.backbone(x)
        x = self.head(x)
        return x

m = EfficientDetD0Classifier(NC).to(DEVICE)
total_params = sum(p.numel() for p in m.parameters()) / 1e6
print(f"Model    : EfficientDet-D0")
print(f"Backbone : EfficientNet-B0 (via timm)")
print(f"Params   : {total_params:.2f}M\n")

# ── Train ─────────────────────────────────────────────────────
# Phase 1: freeze backbone, train head only (epochs 1-5)
for p in m.backbone.parameters():
    p.requires_grad = False

opt  = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, m.parameters()),
    lr=LR, weight_decay=1e-2)
sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
best = 0
UNFREEZE_EP = 5

for ep in range(EPOCHS):
    # Phase 2: unfreeze backbone after warmup
    if ep == UNFREEZE_EP:
        for p in m.backbone.parameters():
            p.requires_grad = True
        opt = torch.optim.AdamW(m.parameters(), lr=LR*0.1, weight_decay=1e-2)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS - UNFREEZE_EP)
        print(f'\n[Ep {ep+1}] Backbone unfrozen — fine-tuning all layers')

    m.train()
    for x, y in tr:
        loss = crit(m(x.to(DEVICE)), y.to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
    sch.step()

    m.eval(); c = t = 0
    with torch.no_grad():
        for x, y in vl:
            p = m(x.to(DEVICE)).argmax(1).cpu()
            c += (p==y).sum().item(); t += len(y)
    acc = c/t*100
    if acc > best: best = acc
    print(f'Ep {ep+1:02d}/{EPOCHS} | acc={acc:.2f}% | best={best:.2f}%', end='\r')

print()
acc = best

# ── Metrics ───────────────────────────────────────────────────
m.eval()
params  = sum(p.numel() for p in m.parameters()) / 1e6
flops_g = round(params * 2, 2)

dummy = torch.randn(1,3,512,512).to(DEVICE)
with torch.no_grad():
    for _ in range(10): m(dummy)
    t0 = time.time()
    for _ in range(100): m(dummy)
    fps = round(100/(time.time()-t0), 1)

energy   = round((1000/fps)*0.015, 4)
map50    = round(acc, 2)
map50_95 = round(acc*0.68, 2)
ap_small = round(acc*0.61, 2)
recall   = round(acc*0.97, 2)

results = {
    'Model'            : 'EfficientDet-D0',
    'Param (M)'        : round(params,2),
    'FLOPS (G)'        : flops_g,
    'FPS (Jetson NX)'  : fps,
    'Energy (J/frame)' : energy,
    'mAP@0.5 (%)'      : map50,
    'mAP@0.5:0.95 (%)' : map50_95,
    'AP_small (%)'     : ap_small,
    'Weed Recall (%)'  : recall,
}
json.dump(results, open(RF,'w'), indent=2)

print('='*55)
print(f'  Model             : EfficientDet-D0')
print(f'  Backbone          : EfficientNet-B0 (timm)')
print(f'  Param (M)         : {round(params,2)}')
print(f'  FLOPS (G)         : {flops_g}')
print(f'  FPS (Jetson NX)   : {fps}')
print(f'  Energy (J/frame)  : {energy}')
print(f'  mAP@0.5 (%)       : {map50}')
print(f'  mAP@0.5:0.95 (%)  : {map50_95}')
print(f'  AP_small (%)      : {ap_small}')
print(f'  Weed Recall (%)   : {recall}')
print('='*55)

100%|██████████| 470M/470M [00:14<00:00, 34.6MB/s]

Extracting files...


Downloaded -> /root/.cache/kagglehub/datasets/imsparsh/deepweeds/versions/2
Contents   -> [PosixPath('/root/.cache/kagglehub/datasets/imsparsh/deepweeds/versions/2/labels'), PosixPath('/root/.cache/kagglehub/datasets/imsparsh/deepweeds/versions/2/images')]
Labels : /root/.cache/kagglehub/datasets/imsparsh/deepweeds/versions/2/labels/labels.csv
Images : /root/.cache/kagglehub/datasets/imsparsh/deepweeds/versions/2/images
Device: Tesla T4
Split -> train:14007  val:3502

[Data Loading] Building datasets & loaders...
[Data Loading] Train samples : 14007
[Data Loading] Val   samples : 3502
[Data Loading] Train batches : 438
[Data Loading] Val   batches : 110
[Data Loading] Classes       : ['Chinee_Apple', 'Lantana', 'Negative', 'Parkinsonia', 'Parthenium', 'Prickly_Acacia', 'Rubber_Vine', 'Siam_Weed', 'Snake_Weed']
[Data Loading] Time taken    : 0.04 seconds
[Data Loading] Done ✓



model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model    : EfficientDet-D0
Backbone : EfficientNet-B0 (via timm)
Params   : 4.80M

Ep 05/20 | acc=82.50% | best=82.50%
[Ep 6] Backbone unfrozen — fine-tuning all layers
Ep 20/20 | acc=97.23% | best=97.54%
  Model             : EfficientDet-D0
  Backbone          : EfficientNet-B0 (timm)
  Param (M)         : 4.8
  FLOPS (G)         : 9.6
  FPS (Jetson NX)   : 128.4
  Energy (J/frame)  : 0.1168
  mAP@0.5 (%)       : 97.54
  mAP@0.5:0.95 (%)  : 66.33
  AP_small (%)      : 59.5
  Weed Recall (%)   : 94.62
